# Перед показом (YingQian) — Демонстрация рекомендаций по мультиагентным фильмам

на основе **HelloAgents** из Pipeline + Tool-use：

1. **портрет Agent**(без инструментов)→ TasteProfile  
2. **Поиск Agent**（TMDB Tool）→ настоящий фильм-кандидат  
3. **рекомендовать Agent**(без инструментов)→ Внесено в белый список + причина  

## Инструкция по применению

1. Сначала настройте `backend/.env`(Доступно с `.env.example` копия)  
2. Начните с корневого каталога проекта. Юпитер, запусти это последовательно Notebook  
3. Необходимо иметь возможность доступа TMDB с твоим LLM API

---

## Нет. 1 Раздел: Подготовка среды

In [ ]:
import os
import sys
from pathlib import Path

# Корневой каталог проекта = Книга notebook каталог
ROOT = Path.cwd().resolve()
BACKEND = ROOT / "backend"
assert (BACKEND / "app").exists(), f"Не найден backend/app — откройте notebook из корня проекта (текущий: {ROOT}"

sys.path.insert(0, str(BACKEND))

# Приоритизация загрузки backend/.env
from dotenv import load_dotenv

env_path = BACKEND / ".env"
if not env_path.exists():
    raise FileNotFoundError(
        f"Не найден {env_path}\n"
        "Выполните: copy .env.example backend\.env и укажите ключи TMDB / LLM"
    )
load_dotenv(env_path)

print("ROOT   :", ROOT)
print("BACKEND:", BACKEND)
print("TMDB   :", "настроен" if (os.getenv("TMDB_ACCESS_TOKEN") or os.getenv("TMDB_API_KEY")) else "отсутствует")
print("LLM    :", "настроен" if os.getenv("LLM_API_KEY") else "отсутствует")
print("MODEL  :", os.getenv("LLM_MODEL_ID") or "(не установлено)")

---

## Нет. 2 Раздел: Создание запроса на рекомендации

In [ ]:
from app.models.schemas import RecommendRequest

request = RecommendRequest(
    mood="расслабиться",
    party_type="один",
    genres=["драма", "комедия"],
    max_runtime_minutes=120,
    region_preference="без ограничений",
    year_preference="последние 10 лет",
    exclude_titles=[],
    spoilers_ok=False,
    free_text="не слишком тяжёлое, подходит для вечера выходного дня",
    exclude_ids=[],
)

print(request.model_dump_json(indent=2, ensure_ascii=False))

---

## Нет. 3 Раздел: Запуск многоагентного конвейера рекомендаций

> Завершите примерно один раз. 40–60 секунд, пожалуйста, подождите терпеливо.

In [ ]:
from app.agents.movie_recommender_agent import MultiAgentMovieRecommender

recommender = MultiAgentMovieRecommender()
result, trace_id = recommender.recommend(request)

print("trace_id:", trace_id)
print("fallback:", result.fallback)
if result.taste_profile:
    print("портретное резюме:", result.taste_profile.summary)
    print("тип подсказки:", result.taste_profile.genre_hints)
print(f"Рекомендуемое количество: {len(result.movies)}")
print("=" * 50)
for i, m in enumerate(result.movies, 1):
    print(f"{i}. {m.title} ({m.year or '?'})  счет={m.rating}")
    print(f"   причина: {m.reason}")
    print(f"   плакат: {m.poster_url}")
    print()

---

## Нет. 4 Часть (необязательно): Только проверка TMDB возможность подключения

In [ ]:
from app.services.movie_service import get_movie_service

svc = get_movie_service()
movies = svc.discover(with_genres="комедия", sort_by="popularity.desc", page=1)
print(f"discover возвращаться {len(movies)} Отделение, фронт 5 отделение:")
for m in movies[:5]:
    print(f"- {m.id} | {m.title} | {m.year} | {m.rating}")

---

## Подвести итог

- Сборочная линия: портреты → Поиск(TMDB Tool) → рекомендовать(id белый список)  
- Web форма:`backend` FastAPI + `frontend` React  
- нравиться TMDB Время ожидания соединения истекло (WinError 10060), пожалуйста, проверьте прокси/VPN Повторите попытку позже